In [1]:
import os
import cv2   
import shutil
import random
from tqdm import tqdm 
from glob import glob  
import tensorflow as tf    
from collections import defaultdict

In [2]:
base_path = r"D:\final year project\all datasets\original\Second Set"
preprocessed_path1 = r"D:\final year project\all datasets\original\resized_set"
preprocessed_path2 = r"D:\final year project\all datasets\original\denoised_set"
preprocessed_path3 = r"D:\final year project\all datasets\original\enhanced_set"
preprocessed_path4 = r"D:\final year project\all datasets\original\original_dataset"
os.makedirs(preprocessed_path1, exist_ok=True)
os.makedirs(preprocessed_path2, exist_ok=True)
os.makedirs(preprocessed_path3, exist_ok=True)
os.makedirs(preprocessed_path4, exist_ok=True)

In [11]:
# RESIZE AUGMENTED IMAGES

TARGET_SIZE = (256, 256)

def resize_dataset(input_dir, output_dir):
    for class_name in ["400x Normal Oral Cavity Histopathological Images", "400x OSCC Histopathological Images"]:
        os.makedirs(os.path.join(output_dir, class_name), exist_ok=True)
        input_path = os.path.join(input_dir, class_name)
        output_path = os.path.join(output_dir, class_name)
        
        img_files = [f for f in os.listdir(input_path) if f.lower().endswith('.jpg')]
        
        for img_file in tqdm(img_files, desc=f"Resizing {class_name}"):
            try:
                img = cv2.imread(os.path.join(input_path, img_file))
                if img is not None:
                    resized = cv2.resize(img, TARGET_SIZE, interpolation=cv2.INTER_AREA)
                    cv2.imwrite(os.path.join(output_path, img_file), resized)
            except Exception:
                continue

print("Resizing images...")
resize_dataset(base_path, preprocessed_path1)
print(f"\nResizing complete. Output saved to: {preprocessed_path1}")

Resizing images...


Resizing 400x Normal Oral Cavity Histopathological Images: 100%|██████████| 201/201 [00:32<00:00,  6.16it/s]
Resizing 400x OSCC Histopathological Images: 100%|██████████| 495/495 [00:49<00:00,  9.93it/s]



Resizing complete. Output saved to: D:\final year project\all datasets\original\resized_set


In [12]:
# Apply median filter to color image
def median_filter_color(img, kernel_size=3):
    # Ensure kernel size is odd and >= 3
    kernel_size = max(3, kernel_size | 1)
    return cv2.medianBlur(img, kernel_size)

# Denoise function using median filter
def denoise_dataset(input_dir, output_dir, kernel_size=3):
    os.makedirs(output_dir, exist_ok=True)
    
    # Get all class folders
    class_folders = [d for d in os.listdir(input_dir) if os.path.isdir(os.path.join(input_dir, d))]
    
    for class_name in tqdm(class_folders, desc="Processing classes"):
        class_input_dir = os.path.join(input_dir, class_name)
        class_output_dir = os.path.join(output_dir, class_name)
        os.makedirs(class_output_dir, exist_ok=True)
        
        # Get all images in class folder
        image_files = [f for f in os.listdir(class_input_dir) if f.lower().endswith(('.jpg'))]
        
        for img_file in tqdm(image_files, desc=f"Processing {class_name}", leave=False):
            img_path = os.path.join(class_input_dir, img_file)
            
            # Load image
            img = cv2.imread(img_path)
            if img is None:
                print(f"Warning: Could not read image {img_path}")
                continue
            
            # Apply median filter
            filtered_img = median_filter_color(img, kernel_size=kernel_size)
            
            # Save result
            output_path = os.path.join(class_output_dir, img_file)
            cv2.imwrite(output_path, filtered_img)

# Example usage
if __name__ == "__main__":
    denoise_dataset(preprocessed_path1, preprocessed_path2, kernel_size=3)

Processing classes:   0%|          | 0/2 [00:00<?, ?it/s]

Processing classes: 100%|██████████| 2/2 [00:22<00:00, 11.35s/it]


In [13]:
#CLAHE

# Create output directory structure
class_dirs = [d for d in os.listdir(preprocessed_path2) if os.path.isdir(os.path.join(preprocessed_path2, d))]
for class_dir in class_dirs:
    os.makedirs(os.path.join(preprocessed_path3, class_dir), exist_ok=True)

# Apply CLAHE using OpenCV
def apply_clahe_to_image(image_np):
    lab = cv2.cvtColor(image_np, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.7, tileGridSize=(8,8))
    l_clahe = clahe.apply(l)
    lab_clahe = cv2.merge((l_clahe, a, b))
    img_clahe = cv2.cvtColor(lab_clahe, cv2.COLOR_LAB2RGB)
    return img_clahe

# Process all images
for class_dir in class_dirs:
    input_class_path = os.path.join(preprocessed_path2, class_dir)
    output_class_path = os.path.join(preprocessed_path3, class_dir)

    image_paths = glob(os.path.join(input_class_path, "*.jpg"))

    for image_path in tqdm(image_paths, desc=f"Processing {class_dir}"):
 
        image = tf.io.read_file(image_path)
        image = tf.image.decode_jpeg(image, channels=3)
        image = tf.image.convert_image_dtype(image, tf.uint8)

        image_np = image.numpy()
        
        # Apply CLAHE
        clahe_image = apply_clahe_to_image(image_np)

        # Save output
        image_name = os.path.basename(image_path)
        output_path = os.path.join(output_class_path, image_name)
        cv2.imwrite(output_path, cv2.cvtColor(clahe_image, cv2.COLOR_RGB2BGR)) 

Processing 400x Normal Oral Cavity Histopathological Images: 100%|██████████| 201/201 [00:04<00:00, 44.88it/s]
Processing 400x OSCC Histopathological Images: 100%|██████████| 495/495 [00:17<00:00, 28.90it/s]


In [3]:
# Set the seed for reproducibility
random.seed(42)

# Paths
SOURCE_DIR = preprocessed_path3
DEST_DIR = preprocessed_path4

# Splits
SPLITS = {
    "train": 0.7,
    "val": 0.15,
    "test": 0.15
}

# Create split folders
for split in SPLITS:
    for class_name in os.listdir(SOURCE_DIR):
        class_path = os.path.join(SOURCE_DIR, class_name)
        if os.path.isdir(class_path):
            split_dir = os.path.join(DEST_DIR, split, class_name)
            os.makedirs(split_dir, exist_ok=True)

# Process each class
for class_name in os.listdir(SOURCE_DIR):
    class_path = os.path.join(SOURCE_DIR, class_name)
    if not os.path.isdir(class_path):
        continue

    images = [f for f in os.listdir(class_path) if f.lower().endswith((".jpg"))]
    random.shuffle(images)

    total = len(images)
    train_end = int(SPLITS["train"] * total)
    val_end = train_end + int(SPLITS["val"] * total)

    split_data = {
        "train": images[:train_end],
        "val": images[train_end:val_end],
        "test": images[val_end:]
    }

    # Copy files with progress bar
    for split, split_images in split_data.items():
        print(f"\nCopying {split} data for class '{class_name}':")
        for img_name in tqdm(split_images, desc=f"{class_name} - {split}", unit="img"):
            src_path = os.path.join(class_path, img_name)
            dest_path = os.path.join(DEST_DIR, split, class_name, img_name)
            shutil.copy2(src_path, dest_path)

print("\n✅ Dataset split completed successfully.")


Copying train data for class '400x Normal Oral Cavity Histopathological Images':


400x Normal Oral Cavity Histopathological Images - train: 100%|██████████| 140/140 [00:02<00:00, 64.27img/s] 



Copying val data for class '400x Normal Oral Cavity Histopathological Images':


400x Normal Oral Cavity Histopathological Images - val: 100%|██████████| 30/30 [00:00<00:00, 111.03img/s]



Copying test data for class '400x Normal Oral Cavity Histopathological Images':


...


Copying val data for class '400x OSCC Histopathological Images':


400x OSCC Histopathological Images - val: 100%|██████████| 74/74 [00:00<00:00, 140.60img/s]



Copying test data for class '400x OSCC Histopathological Images':


400x OSCC Histopathological Images - test: 100%|██████████| 75/75 [00:00<00:00, 123.04img/s]


✅ Dataset split completed successfully.


In [4]:

print("\nImage count in each split/class:")

counts = defaultdict(dict)
for split in SPLITS.keys():
    split_path = os.path.join(DEST_DIR, split)
    for class_name in os.listdir(split_path):
        class_dir = os.path.join(split_path, class_name)
        if os.path.isdir(class_dir):
            num_images = len([
                f for f in os.listdir(class_dir)
                if f.lower().endswith((".jpg"))
            ])
            counts[split][class_name] = num_images

# Display counts
for split in SPLITS.keys():
    print(f"\n{split.upper()}:")
    for class_name, num in counts[split].items():
        print(f"  {class_name}: {num} images")



Image count in each split/class:

TRAIN:
  400x Normal Oral Cavity Histopathological Images: 140 images
  400x OSCC Histopathological Images: 346 images

VAL:
  400x Normal Oral Cavity Histopathological Images: 30 images
  400x OSCC Histopathological Images: 74 images

TEST:
  400x Normal Oral Cavity Histopathological Images: 31 images
  400x OSCC Histopathological Images: 75 images
